# 遺伝子スコアリング

**疾患名**と**遺伝子リスト**を入れると、遺伝子に点数を付けて並べ替えます。

使い方は 3 手順だけです。

1. 「入力」のセルに疾患名と遺伝子リストを書く
2. 上から順に実行する
3. 表が出る。CSV にも保存できる

仕組みの説明や評価・対照実験は、隣の `gene_disease_ranking.ipynb` にあります。
こちらは**使うためのノートブック**です。

---

ひとつだけ、知っておくべき前提があります。このノートブックは
**モデルに遺伝子記号を書かせません**。書かせると `GPR52` が `GPR56` に化けるからです
（どのトークナイザも `GPR52` を 1 かたまりで持たず、`GPR` の続きは学習データでの
出現回数で決まってしまう）。代わりに、渡した記号を「採点」するか、`A`〜`E` の
1 文字だけ答えさせてコード側で記号に戻します。

## 0. 準備

In [ ]:
import os, sys, json, csv, importlib

SKILL = os.path.abspath(os.path.join("..", ".claude", "skills", "gene-disease-ranking"))
if not os.path.isdir(SKILL):
    SKILL = os.path.abspath(os.environ.get("GDR_SKILL_DIR", "."))
sys.path.insert(0, os.path.join(SKILL, "scripts"))

HAVE = {m: importlib.util.find_spec(m) is not None
        for m in ("numpy", "pandas", "torch", "transformers", "llama_cpp")}
print("skill:", SKILL)
print("  " + "  ".join(f"{m}={'y' if ok else 'n'}" for m, ok in HAVE.items()))
if not HAVE["numpy"]:
    print("\nnumpy が必要です:  pip install numpy")

## 1. 入力 — ここだけ書き換える

In [ ]:
# ===== 疾患名 =====
DISEASE = "Cystic fibrosis"

# ===== 遺伝子リスト =====
GENES = [
    "CFTR", "HBB", "HBA1", "HBA2",
    "GPR52", "GPR55", "GPR56", "GPR35",
    "APOE", "APOC1", "HTT", "TP53",
    "SLC6A4", "SLC6A3",
]

# ===== どこで動かすか =====
#   "ollama"       ローカル / Docker の Ollama。段階 2 のみ（下の注意を参照）
#   "llamacpp"     Ollama が持っている GGUF をそのまま使う。段階 1 も動く（推奨）
#   "transformers" 量子化なしの重み
BACKEND = "ollama"
MODEL   = None          # 例: "gemma3:27b" / "EPFLiGHT/Gemma-3-27B-MeditronFO"
OLLAMA_HOST = None      # None なら自動探索（Docker 対応）

# ===== 詰めの設定（最初はそのままで可）=====
TOP_K            = None   # 段階 2 に送る件数。None なら候補数から自動
GROUP_SIZE       = 4      # 1 問あたりの遺伝子数（+ 逃げ道で A〜E）
ROTATIONS        = 4      # 選択肢の順序を何通り試すか
MARGIN_THRESHOLD = 0.5    # 1 位と 2 位の差がこれ未満なら「判定しない」
HGNC_PATH        = None   # 例: "/path/to/hgnc_complete_set.txt"（別名の統一）

print(f"疾患         : {DISEASE}")
print(f"候補         : {len(GENES)} 個")
print(f"バックエンド : {BACKEND}")
print(f"モデル       : {MODEL or '★未設定 — ここを埋めないと採点できません'}")

## 2. 接続の確認

Ollama の場合、ここで**何ができるか**が確定します。

- 段階 2（A〜E で答えさせる）はどの環境でも動きます
- **段階 1（渡した記号を採点する）は Ollama では動きません。** Ollama が返す確率は
  「モデルが生成したトークン」のもので、こちらが渡した文字列を採点する機能が無いためです
- 段階 1 も使いたい場合は `BACKEND = "llamacpp"`。Ollama が既に持っている
  GGUF ファイルをそのまま読むので、**再ダウンロードは不要**です

In [ ]:
ranker = None

if not MODEL:
    print("MODEL が未設定です。1 章の MODEL に名前を入れてください。")
    print("  Ollama なら `ollama list` に出てくる名前（例: gemma3:27b）")
else:
    if BACKEND == "ollama" and OLLAMA_HOST is None:
        from backends import discover_ollama_host
        OLLAMA_HOST = discover_ollama_host()
        if OLLAMA_HOST:
            print(f"Ollama: {OLLAMA_HOST}")
        else:
            print("Ollama が見つかりません。")
            print("  ・起動しているか:            docker ps / ollama serve")
            print("  ・ポートが公開されているか:  -p 11434:11434")
            print("  ・別ホストなら OLLAMA_HOST に直接指定")

    from rank import GeneRanker
    kw = {"host": OLLAMA_HOST} if BACKEND == "ollama" and OLLAMA_HOST else {}
    if HGNC_PATH:
        kw["hgnc"] = HGNC_PATH
    ranker = GeneRanker(MODEL, backend=BACKEND, **kw)
    print(f"backend={BACKEND}  段階 1={'使える' if ranker.supports_pmi else '使えない'}")

## 3. 採点する

`score(疾患名, 遺伝子リスト)` を呼ぶだけです。何度でも呼べます。

In [ ]:
def score(disease, genes, ranker=None, **kw):
    """疾患名と遺伝子リストを受け取り、点数付きで並べ替えて返す。"""
    r = ranker or globals().get("ranker")
    if r is None:
        raise RuntimeError("2 章でモデルを設定してください")
    opts = dict(top_k=TOP_K, group_size=GROUP_SIZE, rotations=ROTATIONS,
                margin_threshold=MARGIN_THRESHOLD)
    opts.update(kw)
    return r.rank(disease=disease, genes=genes, **opts)


def to_rows(result, bar_width=16):
    """結果を、そのまま表にできる行のリストにする。

    段階 1 を使わなかったときは pmi 列を出さない。全部 None の列は
    情報がないだけでなく、あたかも測ったかのように見えて紛らわしい。
    """
    two_stage = bool(result["stage1"])
    pmi = {x["gene"]: x["pmi"] for x in (result["stage1"] or [])}

    vals = [x["score"] for x in result["stage2"]]
    lo, hi = (min(vals), max(vals)) if vals else (0.0, 1.0)
    span = (hi - lo) or 1.0

    rows = []
    for i, x in enumerate(result["stage2"], 1):
        row = {"rank": i, "gene": x["gene"], "score": x["score"]}
        if two_stage:
            row["pmi"] = pmi.get(x["gene"])
        n = int(round((x["score"] - lo) / span * bar_width))
        row["|"] = "#" * n
        rows.append(row)
    return rows


result = score(DISEASE, GENES) if ranker else None
print("完了" if result else "スキップしました（モデル未設定）")

### 結果

In [ ]:
def show(rows, cols=None):
    if not rows:
        print("（該当なし）")
        return
    cols = cols or list(rows[0])
    if HAVE["pandas"]:
        import pandas as pd
        df = pd.DataFrame(rows, columns=cols)
        try:
            display(df)
        except NameError:
            print(df.to_string(index=False))
        return
    w = [max(len(str(c)), *(len(str(r.get(c, ""))) for r in rows)) for c in cols]
    head = "  ".join(str(c).ljust(x) for c, x in zip(cols, w))
    print(head); print("-" * len(head))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(x) for c, x in zip(cols, w)))


if result:
    rows = to_rows(result)
    show(rows)

    print()
    if result["call"]:
        print(f"判定           : {result['call']}")
    else:
        print(f"判定           : なし（{result['abstain_reason']}）")
    print(f"1 位と 2 位の差 : {result['margin']}")
    print(f"順序への頑健さ  : {result['rank_stability']}   1.0 に近いほど良い")
    print(f"使った方法      : {' → '.join(result['plan']['stages'])}")
    for w in result["warnings"]:
        print(f"\n⚠ {w}")
else:
    print("モデル未設定のため結果がありません。")

点数の読み方：

- **score** … 段階 2 の平均対数確率。**同じ疾患の中での順位づけ**に使うもので、
  疾患をまたいだ比較はできません
- **pmi** … 段階 1 の点数（`llamacpp` / `transformers` のときだけ出ます）。
  「疾患名を出したことで、この遺伝子の確率がどれだけ上がったか」。
  出現頻度の分を引いてあるので、TP53 のような有名な遺伝子が上に来にくくなります
- **1 位と 2 位の差が小さいときは間違えている確率が高い**ので、
  無理に答えを出さず「判定なし」になります。当てずっぽうより有用です

## 4. 保存

In [ ]:
OUT_CSV   = "gene_scores.csv"
OUT_JSONL = "gene_scores.jsonl"

if result:
    rows = to_rows(result)
    with open(OUT_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["disease"] + list(rows[0]))
        w.writeheader()
        for r in rows:
            w.writerow({"disease": result["disease"], **r})
    with open(OUT_JSONL, "w") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
    print(f"{OUT_CSV} と {OUT_JSONL} に保存しました")
    print("jsonl は evaluate.py にそのまま渡せます（ranked キーを持っています）")
else:
    print("保存するものがありません。")

## 5. 複数の疾患をまとめて

同じ遺伝子リストを使うなら `ranker` を使い回してください。
疾患に依存しない項を 1 回だけ計算して再利用します。

In [ ]:
DISEASES = [
    "Cystic fibrosis",
    "Huntington disease",
    "Sickle cell disease",
    "Alzheimer disease",
]

results = []
if ranker:
    results = [score(d, GENES) for d in DISEASES]

    show([{"disease": r["disease"],
           "1位": r["stage2"][0]["gene"],
           "score": r["stage2"][0]["score"],
           "2位": r["stage2"][1]["gene"] if len(r["stage2"]) > 1 else "",
           "差": r["margin"],
           "判定": r["call"] or "なし"}
          for r in results])

    with open("gene_scores_all.jsonl", "w") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("\ngene_scores_all.jsonl に保存しました")
else:
    print("モデル未設定のため飛ばしました。")

## 6. 数字を信じる前に

このノートブックは**順位を出すだけ**で、正しさは何も保証しません。使う前に：

1. **対照と比べる。** ランダム、頻度のみ、疾患名を入れ替えたもの、そして
   Open Targets 単独。特に最後は重要で、**これに勝てないなら LLM を外すべき**です。
   Open Targets は遺伝子-疾患の関連づけを既に十分うまくやっており、LLM の価値は
   「文章にはあるが整備済みデータには無い関連」に限られます。
   手順は `gene_disease_ranking.ipynb` の 8 章にあります。

2. **エビデンスで裏を取る。** LLM の点数は仮説であって証拠ではありません。
   PubTator3 や Open Targets で裏付けを取り、裏付けが 0 件のものは
   捨てるのではなく**印を付けて**出します。

3. **規模。** 4 疾患・14 遺伝子では何も言えません。既知のペア 100 組以上で測ってください。

4. **`⚠` の警告を無視しない。** 特に `ollama` を使っていて段階 1 が飛ばされた場合、
   出現頻度の偏りを打ち消すものがありません。どの疾患でも有名な遺伝子が上に来ます。